# [INFO] Glu-Stock: 02_SIGNAL_INFERENCE
**Phase**: Ensemble Intelligence (LightGBM + CNN) | v18.22 (Total Recall)

This notebook performs dual-brain inference using a Weighted Ensemble approach.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib tensorflow python-dotenv ta lightgbm


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Universe)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
    
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})

    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

    def wait_for_queue(self, queue_name: str, max_retries=15, interval=60):
        import time
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs: return self.get_and_clear_queue(queue_name)
            time.sleep(interval)
        return []


In [ ]:
# [BRAIN] SECTION 3: CORE LOGIC (Institutional Predictors & Weighted Ensemble)
import ta

def frac_diff(series, d=0.4, window=100):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    w = np.array(w[::-1])
    result = np.full(len(series), np.nan)
    for t in range(window - 1, len(series)):
        result[t] = np.dot(w, series[t - window + 1:t + 1])
    return result

class MLPredictor:
    def __init__(self, path):
        brain = joblib.load(path)
        self.model = brain.get('model')
        self.features = brain.get('features', [])

    def build_features(self, df):
        close = df['Close'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        volume = df['Volume'].squeeze()
        feat = pd.DataFrame(index=df.index)
        feat['Returns'] = close.pct_change()
        feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
        macd = ta.trend.MACD(close=close)
        feat['MACD'] = macd.macd_diff()
        boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
        feat['BB_High'] = boll.bollinger_hband_indicator()
        feat['BB_Low'] = boll.bollinger_lband_indicator()
        feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
        feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
        obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
        feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
        feat['day_of_week'] = df.index.dayofweek
        feat['week_of_month'] = (df.index.day - 1) // 7
        feat['frac_diff_close'] = frac_diff(close.values.flatten(), d=0.4, window=100)
        vol_ma = volume.rolling(20).mean()
        feat['vol_ratio'] = volume / (vol_ma + 1e-7)
        return feat.dropna()

    def predict(self, df):
        try:
            feat = self.build_features(df)
            if len(feat) == 0: return 0.0
            row = feat[self.features].tail(1)
            proba = self.model.predict_proba(row)[0]
            return float(proba[1])
        except: return 0.0

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()

    def predict(self, df):
        try:
            close = df['Close'].squeeze().values[-30:]
            high = df['High'].squeeze().values[-30:]
            low = df['Low'].squeeze().values[-30:]
            volume = df['Volume'].squeeze().values[-30:]
            opn = df['Open'].squeeze().values[-30:]
            raw = np.column_stack([opn, high, low, close, volume])
            seq_min, seq_max = raw.min(axis=0), raw.max(axis=0)
            norm_seq = (raw - seq_min) / (seq_max - seq_min + 1e-7)
            input_details = self.interpreter.get_input_details()
            input_data = np.expand_dims(norm_seq.astype(np.float32), axis=0)
            self.interpreter.set_tensor(input_details[0]['index'], input_data)
            self.interpreter.invoke()
            output = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(output[1]) if len(output) > 1 else float(output[0])
        except: return 0.5


In [ ]:
# [RUN] SECTION 4: MAIN EXECUTION
def find_model_file(filename):
    search_paths = ['/kaggle/input', '/kaggle/working', '.', 'data/models']
    for root_dir in search_paths:
        if not os.path.exists(root_dir): continue
        for root, dirs, files in os.walk(root_dir):
            if filename in files: return os.path.join(root, filename)
    return None

def run_inference():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    lgbm_path = find_model_file('glu_brain_v1.joblib')
    cnn_path = find_model_file('cnn_daily_t2.tflite')
    if not lgbm_path or not cnn_path: return
    
    queue = fb.wait_for_queue('research')
    if not queue: return
    candidates = []
    for q in queue: 
        if isinstance(q, list): candidates.extend(q)
        else: candidates.append(q)
    
    lgbm = MLPredictor(lgbm_path)
    cnn = CNNPredictor(cnn_path)
    signals = {}
    
    target_tickers = list(set(candidates))
    n_ok, n_total = 0, len(target_tickers)
    
    for ticker in target_tickers:
        try:
            df = yf.download(ticker, period='200d', progress=False, auto_adjust=True)
            if len(df) < 150: continue
            
            l_prob = lgbm.predict(df)
            c_prob = cnn.predict(df)
            
            # WEIGHTED ENSEMBLE: 40% LGBM / 60% CNN
            ensemble_score = (l_prob * 0.4) + (c_prob * 0.6)
            
            if ensemble_score >= 0.5:
                signals[ticker] = {'price': float(df['Close'].iloc[-1]), 'score': ensemble_score, 'timestamp': datetime.now().isoformat()}
                print(f'[SIGNAL] {ticker} APPROVED (Score: {ensemble_score:.2%})')
                n_ok += 1
        except: continue
            
    health = {'candidates': n_total, 'signals': n_ok}
    fb.log_event("INFERENCE_HEALTH", health)
    if signals:
        fb.push_task('signals', signals)
        print(f'[OK] Dispatched {len(signals)} signals.')
    else: print('[BLOCK] No signals met ensemble score.')

run_inference()